# Di cosa parlano i Papi — temi per Papa (parole vs significato)

Ti ricordi le domande di quelle sere a commentare i giornali? *"Francesco è
comunista? Ha rotto con i Papi di prima?"*. Invece di tirare a indovinare,
**guardiamo i dati**: ~25.000 testi dei quattro Papi (Giovanni Paolo II,
Benedetto XVI, Francesco, Leone XIV), e per ogni tema contiamo in quanti
documenti compare.

Due modi di contare, messi a confronto:
- **regex (parole)** — il documento contiene le parole chiave del tema;
- **semantico (embedding)** — il documento è tra i più vicini al *significato* del
  tema, anche senza la parola esatta ("casa comune" parla di ambiente).

Il `lift` dice quanto un Papa sta sopra (>1) o sotto (<1) la media dei quattro.

## Come gira

Legge l'indice LanceDB del *vector database* (repo gemello) tramite `ponte.py`.
Servono l'indice già costruito (`python vdb.py build` nell'altro repo) e
l'ambiente con `sentence-transformers`, `lancedb`, `torch`, `seaborn` (vedi
`../setup/`). Il calcolo embedding gira sulla CPU: un paio di minuti.

In [ ]:
import collections
import numpy as np
import torch  # il matmul va in torch: numpy-BLAS dopo torch può crashare su Windows (MKL)
from ponte import embedder, tabella

tab = tabella()
t = tab.search().limit(10**9).to_arrow()
V = np.stack(t.column("vector").to_pylist()).astype("float32")
testi = t.column("testo").to_pylist()
urls  = t.column("url").to_pylist()
papi  = t.column("papa").to_pylist()

doc_chunks, doc_papa = collections.defaultdict(list), {}
for i, u in enumerate(urls):
    doc_chunks[u].append(i)
    doc_papa[u] = papi[i]
docs = list(doc_chunks)
tot = collections.Counter(doc_papa[u] for u in docs)
print(len(docs), "documenti:", {p.split()[-1]: n for p, n in tot.items()})

## I temi

Due gruppi. I **dominanti** (Dio/Vangelo, pace, famiglia) tornano per tutti e
fanno da contrasto; gli **specifici** sono quelli su cui un Papa "porta il suo" —
compresi i temi nuovi: intelligenza artificiale, cambiamento della Chiesa, valore
della vita, dignità dell'uomo, aborto.

Per ogni tema: `regex` = % di documenti che matchano le parole chiave; `semantico`
= % di documenti tra i top-N più vicini al concetto (N = positivi regex → stesso
volume, si confrontano due distribuzioni **senza soglie**, vedi notebook 02).

In [ ]:
# tema -> (regex parole chiave, concetto per l'embedding)
SPECIFICI = {
    "poveri": (r"pover|emarginat|bisognos",
               "i poveri, gli emarginati, gli ultimi, chi è nel bisogno"),
    "migranti": (r"migrant|rifugiat|profugh|immigrat",
                 "i migranti, i rifugiati, chi fugge dalla guerra e cerca accoglienza"),
    "giustizia": (r"disuguagl|iniqu|giustizia social|sfruttament",
                  "la giustizia sociale, le disuguaglianze, lo sfruttamento, l'iniquità"),
    "lavoro": (r"lavorator|operai|sindacat|disoccupa",
               "il lavoro, i lavoratori, gli operai, la disoccupazione, i diritti del lavoro"),
    "ricchezza/denaro": (r"capitalism|profitt|avidit|idolatria del denaro",
                         "l'idolatria del denaro, l'avidità, un'economia che esclude e scarta"),
    "ambiente": (r"ambient|ecolog|casa comune",
                 "la cura del creato, l'ambiente, l'ecologia, la casa comune"),
    "aborto": (r"aborto|abortiv",
               "l'aborto e la difesa della vita nascente e non ancora nata"),
    "valore della vita": (r"valore della vita|sacralit\w* della vita",
                          "il valore e la sacralità della vita umana, dal concepimento alla morte naturale"),
    "dignità dell'uomo": (r"dignità (dell'uomo|della persona|umana)",
                          "la dignità della persona umana e il valore inviolabile di ogni essere umano"),
    "intelligenza artificiale": (r"intelligenza artificiale|algoritm|robotic",
                                 "l'intelligenza artificiale, gli algoritmi, le tecnologie digitali e i loro rischi"),
    "cambiamento Chiesa": (r"riforma della (chiesa|curia)|sinodal|conversione (pastorale|missionaria)|chiesa in uscita",
                           "il cambiamento e la riforma della Chiesa, la sinodalità, la conversione missionaria"),
}
DOMINANTI = {
    "pace": (r"\bpace\b|guerra", "la pace, la fine delle guerre e dei conflitti, la riconciliazione"),
    "famiglia": (r"famigli", "la famiglia, il matrimonio, gli sposi, i figli"),
    "Dio/Vangelo": (r"Cristo|Vangelo|preghiera", "Dio, Gesù Cristo, il Vangelo, la preghiera, la fede"),
}
TEMI = {**SPECIFICI, **DOMINANTI}
PAPI = ["Papa Giovanni Paolo II", "Papa Benedetto XVI", "Papa Francesco", "Papa Leone XIV"]
AB = ["GP2", "BXVI", "FRA", "LEO"]

In [ ]:
import re

emb = embedder()
Vt = torch.from_numpy(V)

pct_reg, pct_sem = {}, {}     # tema -> [% per Papa]
for tema, (rx_str, concetto) in TEMI.items():
    rx = re.compile(rx_str, re.I)
    reg = {u for u in docs if any(rx.search(testi[i]) for i in doc_chunks[u])}
    N = len(reg)
    sims = (Vt @ torch.from_numpy(emb.query(concetto).astype("float32"))).numpy()
    doc_sim = {u: max(sims[i] for i in doc_chunks[u]) for u in docs}
    sem = set(sorted(docs, key=lambda u: -doc_sim[u])[:N]) if N else set()
    creg = collections.Counter(doc_papa[u] for u in reg)
    csem = collections.Counter(doc_papa[u] for u in sem)
    pct_reg[tema] = [100 * creg[p] / tot[p] for p in PAPI]
    pct_sem[tema] = [100 * csem[p] / tot[p] for p in PAPI]
    print(f"{tema:26} regex " + "".join(f"{x:5.0f}%" for x in pct_reg[tema])
          + "   sem " + "".join(f"{x:5.0f}%" for x in pct_sem[tema]))

## I numeri (regex, % di documenti)

| tema | GP2 | BXVI | FRA | LEO |
|---|--:|--:|--:|--:|
| poveri | 33 | 36 | **46** | 41 |
| migranti | 6 | 6 | **12** | 9 |
| giustizia | 6 | 6 | **10** | 8 |
| lavoro | **12** | 7 | 8 | 6 |
| ricchezza/denaro | 5 | 6 | **9** | 8 |
| ambiente | 18 | 16 | 19 | 18 |
| aborto | 2 | 2 | 1 | 1 |
| valore della vita | 1 | 1 | 1 | 1 |
| dignità dell'uomo | 8 | 8 | 6 | **9** |
| intelligenza artificiale | 0 | 0 | 1 | **8** |
| cambiamento Chiesa | 4 | 5 | 7 | **10** |
| *pace* | 50 | 51 | 51 | *70* |
| *famiglia* | 55 | 55 | 51 | 51 |
| *Dio/Vangelo* | 88 | 89 | 82 | 86 |

(il quadro semantico conferma e in più *alza* Francesco/Leone su poveri, migranti
ed economia — lo dicono anche dove la parola secca non c'è)

## La heatmap

Colore = `lift` (rosso: sopra la media dei quattro; blu: sotto); numero = % di
documenti e ×lift. Due pannelli: a parole (regex) e a significato (embedding).

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

def _lift(perc):
    m = np.mean(perc)
    return [p / m if m else np.nan for p in perc]

temi = list(SPECIFICI) + ["—"] + list(DOMINANTI)
fig, axes = plt.subplots(1, 2, figsize=(12, 8), sharey=True)
for k, (ax, (titolo, pct)) in enumerate(zip(
        axes, [("REGEX (parole)", pct_reg), ("EMBEDDING (significato)", pct_sem)])):
    lift = np.full((len(temi), 4), np.nan)
    annot = np.full((len(temi), 4), "", dtype=object)
    for i, tm in enumerate(temi):
        if tm in pct:
            lift[i] = _lift(pct[tm])
            for j in range(4):
                annot[i, j] = f"{pct[tm][j]:.0f}%\n×{lift[i, j]:.2f}"
    sns.heatmap(lift, ax=ax, cmap="RdBu_r", center=1.0, vmin=0.6, vmax=1.6,
                annot=annot, fmt="", annot_kws={"fontsize": 7.5},
                xticklabels=AB, yticklabels=temi, linewidths=.5, linecolor="white",
                mask=np.isnan(lift), cbar=(k == 1),
                cbar_kws={"label": "lift (1 = come tutti)", "extend": "both"})
    ax.set_title(titolo, fontsize=11)
    ax.xaxis.set_label_position("top"); ax.xaxis.tick_top()
    ax.tick_params(axis="y", rotation=0); ax.tick_params(axis="x", rotation=0)
fig.suptitle("Di cosa parlano i Papi — regex vs embedding\n"
             "colore = lift (rosso: più della media) · numero = % documenti  "
             "(⚠️ IA: tema solo recente, GP2/Benedetto non confrontabili)", fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.94])
fig.savefig("temi_per_papa.png", dpi=150, bbox_inches="tight")
print("salvata temi_per_papa.png")

![heatmap dei temi per Papa](temi_per_papa.png)

## Cosa raccontano i dati

**La continuità è schiacciante.** I temi che dominano sono identici per tutti e
quattro: Dio/Vangelo (~85%), pace (~50%), famiglia (~52%) — lift attorno a 1 per
tutti. Su questo i Papi sono la stessa voce. Francesco non è un corpo estraneo.

**Il "comunista".** Sì, Francesco accentua davvero poveri, migranti e
disuguaglianze (lift 1,2–1,5): è il suo timbro, e il semantico lo marca ancora di
più. Ma sono temi **minori** rispetto a Dio/pace/famiglia, e **li trattano tutti**
i Papi — è la dottrina sociale della Chiesa, vecchia di oltre un secolo, non
comunismo. La prova del nove è la riga **lavoro/operai**: il primo è **Giovanni
Paolo II** (lift 1,60, il doppio di Francesco) — proprio il Papa che ha fatto
*cadere* il comunismo. Parlare di poveri e operai non rende comunisti.

**Le firme dei due "nuovi".** L'**intelligenza artificiale** è la firma di Leone
(8%): su GP2/Benedetto è ~0, ma è *temporalmente impossibile*, non una scelta — il
lift lì non va letto. Il **cambiamento della Chiesa** cresce verso Francesco e
Leone (sinodalità, riforma).

**Detto in modi diversi.** Sull'**aborto** la regex premia GP2/Benedetto (parola
esplicita), ma il semantico riavvicina Francesco/Leone: lo dicono altrimenti
("difesa della vita"). È la lezione di metodo del notebook 02 — i temi sfumati si
misurano col *significato*, non con le stringhe.

---
*Nota onesta: qui guardiamo solo numeri e temi **aggregati**, non ripubblichiamo i
testi (sono © Libreria Editrice Vaticana, l'uso è personale e di studio; la fonte
`url` resta nei metadati). In una riga: **Francesco cambia gli accenti, non la
sostanza. Continuità piena, comunismo no.***